# BirdCLEF+ 2026 SED Submission

EXP-004 の学習済みモデル (best_fold0.pt) で推論→提出。

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Data | `birdclef-2026` (Competition) + SED学習Output (Private Dataset) |
| Accelerator | GPU T4 x2 |
| Internet | **OFF** |

In [ ]:
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm
import torchaudio.transforms as T
from torch.cuda.amp import autocast

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# PATHS - SED学習Outputのパスを自動検出
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA, 'Competition data not found'

# 学習済みモデルの検索 (Private Datasetとして追加されている想定)
MODEL_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'best_fold0.pt' in files:
        MODEL_DIR = root
        break
assert MODEL_DIR, 'best_fold0.pt not found - Add SED training output as Input'

TEST_DIR = f'{COMP_DATA}/test_soundscapes'
SAMPLE_SUB = f'{COMP_DATA}/sample_submission.csv'

print(f'Competition data: {COMP_DATA}')
print(f'Model dir: {MODEL_DIR}')
print(f'Test dir: {TEST_DIR}')
print(f'Files in model dir:')
for f in os.listdir(MODEL_DIR):
    print(f'  {f}')

In [ ]:
# CONFIG - 学習時と同じ設定
SR = 32000           # Hz: サンプリング周波数
DURATION = 5.0       # sec: 推論窓長
WINDOW_SAMPLES = int(SR * DURATION)  # 160,000 samples per window
N_MELS = 256
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 0
FMAX = 16000
TOP_DB = 80.0
NUM_CLASSES = 234
IN_CHANNELS = 3
BACKBONE = 'tf_efficientnet_b0.ns_jft_in1k'
TARGET_SIZE = (256, 256)

print(f'Window: {DURATION}s = {WINDOW_SAMPLES} samples')

In [ ]:
# MODEL - 学習時と同じ定義
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise = (att * cls).sum(dim=-1)
        return {'clipwise_logit': clipwise, 'framewise_logit': cls.permute(0, 2, 1)}

class SEDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            BACKBONE, pretrained=False, in_chans=IN_CHANNELS,
            features_only=False, global_pool='', num_classes=0)
        self.gem_pool = GEMFreqPool(p_init=3.0)
        self.head = AttentionSEDHead(self.backbone.num_features, NUM_CLASSES, 0.1)
    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))

# MEL TRANSFORM
class MelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
        self.db = T.AmplitudeToDB(stype='power', top_db=TOP_DB)
        self.resize = torchvision.transforms.Resize(TARGET_SIZE, antialias=True)
    @torch.no_grad()
    def forward(self, waveforms):
        mel = self.db(self.mel(waveforms))
        mel = self.resize(mel)
        B = mel.shape[0]
        flat = mel.reshape(B, -1)
        mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mn) / (mx - mn + 1e-7)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)

print('Model + Mel defined')

In [ ]:
# LOAD MODEL
model = SEDModel().to(device)
ckpt = torch.load(f'{MODEL_DIR}/best_fold0.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Model loaded: epoch={ckpt["epoch"]}')
if 'val_score' in ckpt:
    print(f'  val_cmap={ckpt["val_score"]:.4f}')
if 'train_loss' in ckpt:
    print(f'  train_loss={ckpt["train_loss"]:.4f}')

mel_transform = MelSpectrogramTransform().to(device)

# Label encoder
with open(f'{MODEL_DIR}/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
print(f'Label encoder: {len(le.classes_)} classes')

# Species columns from sample_submission
sample_sub = pd.read_csv(SAMPLE_SUB)
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'Submission: {len(sample_sub)} rows, {len(species_cols)} species')

In [ ]:
# INFERENCE
# test_soundscapes のパスを柔軟に検索
test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))
if not test_files:
    test_files = sorted(glob.glob(f'{TEST_DIR}/**/*.ogg', recursive=True))
if not test_files:
    test_files = sorted(glob.glob(f'{COMP_DATA}/**/*.ogg', recursive=True))
    test_files = [f for f in test_files if 'train_audio' not in f]

print(f'Test files: {len(test_files)}')
if test_files:
    print(f'  First: {test_files[0]}')
    print(f'  Last:  {test_files[-1]}')
else:
    print('  No test files found - will create dummy submission')

rows = []
for fi, fpath in enumerate(test_files):
    stem = os.path.splitext(os.path.basename(fpath))[0]

    try:
        full_wf, _ = librosa.load(fpath, sr=SR, mono=True)
    except Exception as e:
        print(f'  Error loading {stem}: {e}')
        continue

    # 5秒窓でスライド (non-overlapping)
    n_chunks = max(1, len(full_wf) // WINDOW_SAMPLES)
    # 音声をちょうど n_chunks * WINDOW_SAMPLES にリサイズ
    full_wf = full_wf[:n_chunks * WINDOW_SAMPLES]
    if len(full_wf) < n_chunks * WINDOW_SAMPLES:
        full_wf = np.pad(full_wf, (0, n_chunks * WINDOW_SAMPLES - len(full_wf)))

    for i in range(n_chunks):
        start = i * WINDOW_SAMPLES
        window = full_wf[start:start + WINDOW_SAMPLES]

        wf_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad(), autocast():
            mel = mel_transform(wf_tensor)
            out = model(mel)
            probs = torch.sigmoid(out['clipwise_logit']).cpu().numpy()[0]

        # row_id = {stem}_{end_sec} (aidensong123方式: 終了秒)
        end_sec = (i + 1) * int(DURATION)
        row_id = f'{stem}_{end_sec}'

        row = {'row_id': row_id}
        for si, sp in enumerate(le.classes_):
            row[sp] = float(probs[si])
        rows.append(row)

    if (fi + 1) % 10 == 0 or (fi + 1) == len(test_files):
        print(f'  [{fi+1}/{len(test_files)}] {stem} -> {len(rows)} rows total', flush=True)

print(f'\nTotal predictions: {len(rows)}')

In [ ]:
# CREATE SUBMISSION
if rows:
    pred_df = pd.DataFrame(rows)
    submission = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
else:
    # テスト音声なし → sample_submission をそのまま 0.0 で提出
    submission = sample_sub.copy()

# 未カバー種は 0.0
for col in species_cols:
    if col not in submission.columns:
        submission[col] = 0.0
submission = submission.fillna(0.0)

# 列順を合わせる
submission = submission[['row_id'] + species_cols]

# 保存
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
print(f'Saved to /kaggle/working/submission.csv')
print(f'\nSample:')
print(submission.head())
print(f'\nNon-zero predictions per row (mean): {(submission[species_cols] > 0.01).sum(axis=1).mean():.1f}')